In [ ]:
import sys
sys.path.append('D:/cxr-triage')

import torch
import pandas as pd
from torch.utils.data import DataLoader

from src.data.dataset import ChestXrayDataset
from src.data.transforms import get_val_transforms
from src.models.convnext import ConvNeXtModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_ROOT = "F:/X ray dataset/Second Version"
CHECKPOINT_PATH = 'D:/cxr-triage/checkpoints/convv_focal_fixed.pth'  # adjust filename if different

model = ConvNeXtModel(num_classes=14, pretrained=False).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Model loaded")

val_df = pd.read_csv('D:/cxr-triage/data/processed/val.csv')
val_dataset = ChestXrayDataset(csv_path=None, image_root=IMAGE_ROOT, transform=get_val_transforms(image_size=224))
val_dataset.df = val_df
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

with torch.no_grad():
    images, targets = next(iter(val_loader))
    logits = model(images.to(DEVICE))
    probs = torch.sigmoid(logits)

print("Logit min/max/mean:", logits.min().item(), logits.max().item(), logits.mean().item())
print("Logit std per class:", logits.std(dim=0))
print("\nProb min/max/mean:", probs.min().item(), probs.max().item(), probs.mean().item())

PermissionError: [Errno 13] Permission denied: 'D:/cxr-triage/checkpoints/convv_focal_fixed'